# Quantization Aware Training

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load Baseline Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## Define Model with Quantization Aware Training


In [ ]:
q_aware_model = tfmot.quantization.keras.quantize_model(model)

q_aware_model.compile(optimizer='adam',
                      loss=keras.losses.SparseCategoricalCrossentropy(),
                      metrics=['accuracy'])

q_aware_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 28, 28, 1)         3         
 yer)                                                            
                                                                 
 quant_conv2d (QuantizeWrap  (None, 26, 26, 32)        387       
 perV2)                                                          
                                                                 
 quant_max_pooling2d (Quant  (None, 13, 13, 32)        1         
 izeWrapperV2)                                                   
                                                                 
 quant_conv2d_1 (QuantizeWr  (None, 11, 11, 16)        4659      
 apperV2)                                                        
                                                                 
 quant_max_pooling2d_1 (Qua  (None, 5, 5, 16)          1

## Fine-tune for QAT (training data 의 subset을 이용)

In [ ]:
train_images_subset = train_images
train_labels_subset = train_labels

q_aware_model.fit(train_images_subset, train_labels_subset, batch_size=500, epochs=1, validation_split=0.1)

108/108 [==============================] - 6s 15ms/step - loss: 0.0132 - accuracy: 0.9968 - val_loss: 0.0337 - val_accuracy: 0.9915


## LiteRT 모델로 변환 (실제 weight의 양자화 적용, Static Quantization))

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
tflite_qat_file = save_dir + 'mnist_qat.tflite'
open(tflite_qat_file, 'wb').write(tflite_model)

63888

## File size 비교 : Baseline model vs QAT model

In [ ]:
import os
tflite_baseline_model_file = save_dir + 'mnist_baseline_model.tflite'
print("Size of Baseline LiteRT Model file : {}".format(os.path.getsize(tflite_baseline_model_file)))
print("Size of QAT LiteRT Model file : {}".format(os.path.getsize(tflite_qat_file)))

Size of Baseline LiteRT Model file : 233596
Size of QAT LiteRT Model file : 63888


## LiteRT 설치 및 Interpreter 로딩

In [ ]:
!pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 93.4 MB/s eta 0:00:00


In [ ]:
from ai_edge_litert.interpreter import Interpreter

## interpreter 생성 (Baseline model)

In [ ]:
interpreter_base = Interpreter(model_path=str(tflite_baseline_model_file))
interpreter_base.allocate_tensors()

## interpreter 생성 (QAT)

In [ ]:
interpreter_qat = Interpreter(model_path=str(tflite_qat_file))
interpreter_qat.allocate_tensors()

## input/output dtype 확인 (QAT)

In [ ]:
input_dtype = interpreter_qat.get_input_details()[0]['dtype']
output_dtype = interpreter_qat.get_output_details()[0]['dtype']

print("input dtype : {}".format(input_dtype))
print("output dtype : {}".format(output_dtype))

input dtype : <class 'numpy.float32'>
output dtype : <class 'numpy.float32'>


## 추론 실행 (QAT)

In [ ]:
test_input = np.expand_dims(test_images[0],axis=0)

interpreter_qat.set_tensor(interpreter_qat.get_input_details()[0]['index'], test_input)
interpreter_qat.invoke()

prediction = interpreter_qat.get_tensor(interpreter_qat.get_output_details()[0]['index'])

print(prediction)
print(np.argmax(prediction))
print(test_labels[0])

[[0.         0.         0.         0.         0.         0.
  0.         0.99609375 0.         0.        ]]
7
7


## Test data 기반 accuracy 평가

In [ ]:
def evaluate_model(interpreter):
  input_index = interpreter.get_input_details()[0]["index"]
  output_index = interpreter.get_output_details()[0]["index"]

  prediction_digits = []
  for i, test_image in enumerate(test_images):
    test_image = np.expand_dims(test_image, axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

In [ ]:
print(evaluate_model(interpreter_base))
print(evaluate_model(interpreter_qat))

0.9904
0.9911
